In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
import pandas as pd
import numpy as np
# Define paths to your files
train_path = '/content/drive/MyDrive/train_ml03.csv'
test_path = '/content/drive/MyDrive/test_pred_ml03.csv'

# Load files into DataFrames
train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

# Preview the data
print("Train Data:")
print(train_df.head())

print("\nTest Predictions Data:")
print(test_df.head())

Train Data:
   subject_id gender                     race arrival_transport  temperature  \
0    90000355      F                  UNKNOWN         AMBULANCE         97.9   
1    90000004      F         UNABLE TO OBTAIN         AMBULANCE         97.0   
2    90000021      M  HISPANIC/LATINO - CUBAN         AMBULANCE         94.1   
3    90000176      F                  UNKNOWN         AMBULANCE          NaN   
4    90000200      M                    WHITE         AMBULANCE         94.5   

   heartrate  resprate  o2sat   sbp   dbp  ...  vs_sbp_max  vs_sbp_mean  \
0      112.0      28.0   89.0  93.0  46.0  ...        89.0         85.5   
1        NaN       0.0   57.0   NaN   NaN  ...        45.0         45.0   
2        0.0       0.0    NaN   NaN   NaN  ...        24.0         24.0   
3        NaN       NaN    NaN   NaN   NaN  ...        26.0         26.0   
4        0.0       0.0   27.0   NaN   NaN  ...         NaN          NaN   

   vs_o2sat_min  vs_o2sat_max  vs_o2sat_mean  vs_tempera

## Phase 2: Handling Missing Values

### Step 1 — Create Missingness Flags

Before touching any actual values, create binary `_missing` columns (1 = was missing,
0 = was present) for every numeric vital and encounter field that has real gaps in the
training data.

**Why:** EDA showed missingness is not random here — it's MNAR (missing not at random).
For example, `pain_score` is missing for ~100% of BLACK patients and only ~10% of GREEN/
YELLOW, because unresponsive patients physically can't report pain. Flagging this
explicitly lets the model use "this was missing" as a feature in its own right, before
we decide what (if anything) to do with the missing value underneath.

**Note:** Flags are only created for columns that actually have missing values in
`train_df`, so we don't clutter the feature set with useless all-zero flags.

In [5]:
candidate_flag_cols = [
    'pain_score', 'sbp', 'dbp', 'o2sat', 'heartrate', 'resprate',
    'temperature', 'pain_assessable',
    'vs_heartrate_min', 'vs_heartrate_max', 'vs_heartrate_mean',
    'vs_resprate_min', 'vs_resprate_max', 'vs_resprate_mean',
    'vs_sbp_min', 'vs_sbp_max', 'vs_sbp_mean',
    'vs_o2sat_min', 'vs_o2sat_max', 'vs_o2sat_mean',
    'vs_temperature_max'
]

flagged_cols = []
for col in candidate_flag_cols:
    if col in train_df.columns and train_df[col].isna().sum() > 0:
        train_df[f'{col}_missing'] = train_df[col].isna().astype(int)
        if col in test_df.columns:
            test_df[f'{col}_missing'] = test_df[col].isna().astype(int)
        flagged_cols.append(col)

print(f"Created missingness flags for {len(flagged_cols)} columns: {flagged_cols}")

Created missingness flags for 21 columns: ['pain_score', 'sbp', 'dbp', 'o2sat', 'heartrate', 'resprate', 'temperature', 'pain_assessable', 'vs_heartrate_min', 'vs_heartrate_max', 'vs_heartrate_mean', 'vs_resprate_min', 'vs_resprate_max', 'vs_resprate_mean', 'vs_sbp_min', 'vs_sbp_max', 'vs_sbp_mean', 'vs_o2sat_min', 'vs_o2sat_max', 'vs_o2sat_mean', 'vs_temperature_max']


### Step 2 — Leave Numeric Gaps as Real NaN (No Imputation)

Vitals, GCS fields, `vs_*` encounter columns, and count fields are left with their
missing values intact — no median, mean, or other fill value is computed or applied.

**Why:** We initially considered filling with a train-only median, but rejected it —
imputing something like `heartrate` for a BLACK patient (true median = 0, i.e. no
measurable pulse) with a global median (~95) would erase the exact signal that makes
BLACK separable in the first place. Since the data is MNAR, a single fill value doesn't
recover missing information — it manufactures misleading information.

**How this is handled downstream:** Tree-based models (LightGBM/XGBoost/CatBoost) split
on missingness natively, so passing real `NaN` straight through is both simpler and more
information-preserving than any manual fill strategy.

In [6]:
numeric_cols_left_as_nan = [
    'temperature', 'heartrate', 'resprate', 'o2sat', 'sbp', 'dbp', 'pain_score',
    'vs_heartrate_min', 'vs_heartrate_max', 'vs_heartrate_mean',
    'vs_resprate_min', 'vs_resprate_max', 'vs_resprate_mean',
    'vs_sbp_min', 'vs_sbp_max', 'vs_sbp_mean',
    'vs_o2sat_min', 'vs_o2sat_max', 'vs_o2sat_mean',
    'vs_temperature_max', 'n_diagnoses', 'n_home_meds', 'n_vitalsign_readings',
    'gcs_eye', 'gcs_verbal', 'gcs_motor', 'gcs_total',
    'avpu_ordinal', 'follows_commands', 'bp_unobtainable', 'pain_assessable'
]

print("\n=== NaN counts left untouched (intentional — model handles natively) ===")
for col in numeric_cols_left_as_nan:
    if col in train_df.columns:
        n_missing = train_df[col].isna().sum()
        if n_missing > 0:
            print(f"{col}: {n_missing} missing ({n_missing/len(train_df):.1%})")



=== NaN counts left untouched (intentional — model handles natively) ===
temperature: 82 missing (12.3%)
heartrate: 80 missing (12.0%)
resprate: 62 missing (9.3%)
o2sat: 119 missing (17.9%)
sbp: 141 missing (21.2%)
dbp: 142 missing (21.4%)
pain_score: 237 missing (35.7%)
vs_heartrate_min: 31 missing (4.7%)
vs_heartrate_max: 31 missing (4.7%)
vs_heartrate_mean: 31 missing (4.7%)
vs_resprate_min: 17 missing (2.6%)
vs_resprate_max: 17 missing (2.6%)
vs_resprate_mean: 17 missing (2.6%)
vs_sbp_min: 83 missing (12.5%)
vs_sbp_max: 83 missing (12.5%)
vs_sbp_mean: 83 missing (12.5%)
vs_o2sat_min: 71 missing (10.7%)
vs_o2sat_max: 71 missing (10.7%)
vs_o2sat_mean: 71 missing (10.7%)
vs_temperature_max: 37 missing (5.6%)
pain_assessable: 140 missing (21.1%)


### Step 3 — Fill Categorical Columns with 'UNKNOWN'

For text/categorical columns (`gender`, `race`, `arrival_transport`, `avpu`,
`consciousness_source`, `chiefcomplaint`), missing values are filled with an explicit
`'UNKNOWN'` label rather than left as NaN or filled with the most common category.

**Why this is a different rule from Step 2:** this isn't about MNAR bias — it's a
practical necessity, since most modeling libraries need string/category columns to hold
an actual value, not a raw blank. Using `'UNKNOWN'` instead of the majority class also
keeps this consistent with Step 1's logic: missing documentation is itself often
informative (e.g. race is more often undocumented for unresponsive patients) rather than
something to paper over.

In [7]:
# ============================================================
# STEP 3: Categorical columns — keep missing as its own category
# (unrelated to the imputation decision above — categoricals
# still need an explicit label since GBMs need encoded categories,
# not raw NaN, for string/object columns)
# ============================================================
categorical_cols = ['gender', 'race', 'arrival_transport',
                     'avpu', 'consciousness_source', 'chiefcomplaint']

for col in categorical_cols:
    if col in train_df.columns:
        train_df[col] = train_df[col].fillna('UNKNOWN')
    if col in test_df.columns:
        test_df[col] = test_df[col].fillna('UNKNOWN')

print(f"\nFilled categorical NaNs with 'UNKNOWN' for: {categorical_cols}")


Filled categorical NaNs with 'UNKNOWN' for: ['gender', 'race', 'arrival_transport', 'avpu', 'consciousness_source', 'chiefcomplaint']


### Phase 2 Summary

- Missingness flags created for every numeric field with real gaps (signal preserved,
  not discarded)
- Numeric vitals and encounter fields left as genuine `NaN` — no imputation, since the
  missingness pattern is MNAR and a fill value would distort the severity signal
- Categorical fields given an explicit `'UNKNOWN'` label so no column is left with raw
  blanks going into modeling

No value here is invented, and nothing is derived from `test_df` — the only thing
happening is labeling what's missing and leaving genuinely missing numbers as missing,
ready for a model that can handle NaN natively.

**Next:** Phase 3 — feature engineering.

## Phase 3: Feature Engineering

### Step 1 — Vital Sign Abnormality Flags

Convert raw vitals into clinically standard abnormality flags (tachycardia, hypoxia,
hypotension, tachypnea, fever, etc.) using established medical thresholds.

**Why:** With only ~664 rows, the model shouldn't have to rediscover "heart rate > 100
matters" purely from raw numbers — encoding known clinical thresholds directly gives it
a head start.

**Rule:** If the source vital is missing, the flag is also left as `NaN` (not `0`) —
missing means "unknown," not "normal." Filling it with 0 would falsely imply the vital
was checked and came back fine.

In [8]:
def add_vital_flags(df):
    df = df.copy()

    # Heart rate: normal ~60-100 bpm
    df['flag_tachycardic'] = np.where(df['heartrate'].isna(), np.nan, (df['heartrate'] > 100).astype(float))
    df['flag_bradycardic'] = np.where(df['heartrate'].isna(), np.nan, (df['heartrate'] < 60).astype(float))

    # O2 saturation: hypoxia below 92%
    df['flag_hypoxic'] = np.where(df['o2sat'].isna(), np.nan, (df['o2sat'] < 92).astype(float))

    # Systolic BP: hypotension below 90
    df['flag_hypotensive'] = np.where(df['sbp'].isna(), np.nan, (df['sbp'] < 90).astype(float))

    # Respiratory rate: normal ~12-20 breaths/min
    df['flag_tachypneic'] = np.where(df['resprate'].isna(), np.nan, (df['resprate'] > 20).astype(float))
    df['flag_bradypneic'] = np.where(df['resprate'].isna(), np.nan, (df['resprate'] < 12).astype(float))

    # Temperature (Fahrenheit): fever above 100.4, hypothermia below 95
    df['flag_fever'] = np.where(df['temperature'].isna(), np.nan, (df['temperature'] > 100.4).astype(float))
    df['flag_hypothermic'] = np.where(df['temperature'].isna(), np.nan, (df['temperature'] < 95).astype(float))

    return df

train_df = add_vital_flags(train_df)
test_df = add_vital_flags(test_df)

print("Added 8 vital abnormality flags")
print(train_df[[c for c in train_df.columns if c.startswith('flag_')]].mean())

Added 8 vital abnormality flags
flag_tachycardic    0.339041
flag_bradycardic    0.138699
flag_hypoxic        0.332110
flag_hypotensive    0.170172
flag_tachypneic     0.373754
flag_bradypneic     0.136213
flag_fever          0.005155
flag_hypothermic    0.072165
dtype: float64


### Step 2 — Vital Sign Instability (Range Features)

Compute the spread (max − min) for heart rate, respiratory rate, systolic BP, and O2 sat
across the encounter, using the `vs_*` columns we chose to keep.

**Why:** A patient whose heart rate swung from 70 to 160 during their stay is clinically
very different from one who stayed steady at 90 — but that difference is currently split
across two separate min/max columns instead of expressed directly. This feature surfaces
instability as a single, explicit number.

**Note:** If either the min or max value is missing, the resulting range is `NaN` too —
this propagates naturally and requires no extra handling.

In [9]:
def add_vital_ranges(df):
    df = df.copy()

    vs_pairs = [
        ('vs_heartrate_min', 'vs_heartrate_max', 'vs_heartrate_range'),
        ('vs_resprate_min', 'vs_resprate_max', 'vs_resprate_range'),
        ('vs_sbp_min', 'vs_sbp_max', 'vs_sbp_range'),
        ('vs_o2sat_min', 'vs_o2sat_max', 'vs_o2sat_range'),
    ]

    for min_col, max_col, new_col in vs_pairs:
        if min_col in df.columns and max_col in df.columns:
            df[new_col] = df[max_col] - df[min_col]  # NaN propagates naturally if either side is missing

    return df

train_df = add_vital_ranges(train_df)
test_df = add_vital_ranges(test_df)

print("Added 4 vital instability range features")
print(train_df[['vs_heartrate_range','vs_resprate_range','vs_sbp_range','vs_o2sat_range']].describe())

Added 4 vital instability range features
       vs_heartrate_range  vs_resprate_range  vs_sbp_range  vs_o2sat_range
count          633.000000         647.000000    581.000000      593.000000
mean            19.262816           3.827468     18.754811        3.621190
std             22.608191           3.870246     15.959311        3.694083
min              0.000000           0.000000      0.000000        0.000000
25%              4.000000           1.000000      7.000000        1.000000
50%             14.000000           3.000000     16.000000        3.000000
75%             24.000000           6.000000     27.000000        5.000000
max            129.000000          26.000000    108.000000       38.000000


### Step 3 — Consolidated Consciousness Signal

Add one summary flag — `flag_altered_consciousness` — derived from `avpu`
(1 if anything other than "Alert", else 0).

**Why:** EDA showed `gcs_total`, `avpu_ordinal`, and `follows_commands` are almost the
same variable (correlation ≥ 0.86 with each other). Rather than dropping two of them
and losing information, we keep all three for the model and add this one clean,
human-readable flag on top — it becomes the headline feature for the interpretability


In [10]:
def add_consciousness_flag(df):
    df = df.copy()
    # 'A' (Alert) on AVPU is the only fully normal state — anything else is altered consciousness
    df['flag_altered_consciousness'] = np.where(
        df['avpu'].isna(), np.nan,
        (df['avpu'] != 'A').astype(float)
    )
    return df

train_df = add_consciousness_flag(train_df)
test_df = add_consciousness_flag(test_df)

print("Added altered_consciousness flag")
print(train_df['flag_altered_consciousness'].value_counts(dropna=False))

Added altered_consciousness flag
flag_altered_consciousness
0.0    482
1.0    182
Name: count, dtype: int64


### Step 4 — Total Missingness Count

Sum every `_missing` flag column created in Phase 2 into a single
`total_missing_count` per patient.

**Why:** EDA confirmed missingness itself correlates strongly with severity — e.g.
BLACK patients are missing nearly every vital (unresponsive patients can't be fully
assessed). This feature compresses that pattern, currently spread across a dozen
separate flag columns, into one compact severity proxy.

In [11]:
def add_missingness_count(df):
    df = df.copy()
    missing_flag_cols = [c for c in df.columns if c.endswith('_missing')]
    df['total_missing_count'] = df[missing_flag_cols].sum(axis=1)
    return df, missing_flag_cols

train_df, missing_flag_cols = add_missingness_count(train_df)
test_df, _ = add_missingness_count(test_df)

print(f"Summed {len(missing_flag_cols)} missingness flags into total_missing_count")
print(train_df['total_missing_count'].describe())

Summed 21 missingness flags into total_missing_count
count    664.000000
mean       2.478916
std        4.388334
min        0.000000
25%        0.000000
50%        0.000000
75%        2.000000
max       21.000000
Name: total_missing_count, dtype: float64


### Step 5 — Bucket `chiefcomplaint` into Clinical Categories

Group the 175 unique raw complaint strings into ~8 broad clinical categories
(ARREST_UNRESPONSIVE, CARDIAC, RESPIRATORY, TRAUMA, GI, NEURO, INFECTIOUS, PAIN_OTHER,
OTHER) using keyword matching.

**Why:** 175 unique values is too sparse to one-hot directly against only 664 rows —
most categories would have just a handful of examples. Bucketing preserves the strong
signal EDA found (e.g. "S/P ARREST" clustering in BLACK, "SOB"/"SEPSIS" in RED) while
cutting dimensionality drastically.

In [12]:
def bucket_complaint(complaint):
    if pd.isna(complaint):
        return 'UNKNOWN'
    c = str(complaint).upper()

    if any(k in c for k in ['ARREST', 'UNRESPONSIVE', 'FOUND DOWN', 'CODE']):
        return 'ARREST_UNRESPONSIVE'
    elif any(k in c for k in ['CHEST PAIN', 'CARDIAC', 'PALPITATION']):
        return 'CARDIAC'
    elif any(k in c for k in ['SOB', 'RESP', 'BREATH', 'COUGH']):
        return 'RESPIRATORY'
    elif any(k in c for k in ['TRAUMA', 'MVC', 'FALL', 'FRACTURE', 'INJURY', 'LACERATION']):
        return 'TRAUMA'
    elif any(k in c for k in ['BLEED', 'GI ', 'ABD PAIN', 'NAUSEA', 'VOMIT']):
        return 'GI'
    elif any(k in c for k in ['HEAD', 'DIZZ', 'SEIZURE', 'STROKE', 'WEAKNESS']):
        return 'NEURO'
    elif any(k in c for k in ['SEPSIS', 'FEVER', 'INFECTION']):
        return 'INFECTIOUS'
    elif any(k in c for k in ['PAIN', 'HEADACHE', 'BACK']):
        return 'PAIN_OTHER'
    else:
        return 'OTHER'

train_df['complaint_category'] = train_df['chiefcomplaint'].apply(bucket_complaint)
test_df['complaint_category'] = test_df['chiefcomplaint'].apply(bucket_complaint)

print("Complaint bucketing result:")
print(train_df['complaint_category'].value_counts())

Complaint bucketing result:
complaint_category
TRAUMA                 160
OTHER                  121
GI                      81
NEURO                   80
ARREST_UNRESPONSIVE     80
PAIN_OTHER              50
CARDIAC                 43
RESPIRATORY             30
INFECTIOUS              19
Name: count, dtype: int64


### Step 6 — Finalize Categorical Dtypes for Modeling

Convert all remaining string/object columns to pandas `category` dtype so LightGBM/
CatBoost can consume them natively, without manual one-hot encoding.

**Why:** Native categorical support plays correctly with the NaN-passthrough decision
from Phase 2, and avoids the dimensionality blow-up of one-hot encoding on a small dataset.

**Caution:** Test-set categories are aligned to the categories seen in `train_df`. Any
category value in `test_df` that never appeared in `train_df` will become `NaN` after
this step — the code checks for and flags this explicitly so it isn't missed silently.

In [13]:
categorical_cols_final = ['gender', 'race', 'arrival_transport', 'avpu',
                           'consciousness_source', 'complaint_category']

for col in categorical_cols_final:
    if col in train_df.columns:
        train_df[col] = train_df[col].astype('category')
    if col in test_df.columns:
        # align test categories to train's known categories — anything unseen becomes NaN, flagged below
        test_df[col] = pd.Categorical(test_df[col], categories=train_df[col].cat.categories)

# Check for unseen categories that got silently turned into NaN
for col in categorical_cols_final:
    if col in test_df.columns:
        n_unseen = test_df[col].isna().sum() - (train_df[col].isna().sum() if col in train_df.columns else 0)
        if test_df[col].isna().sum() > 0:
            print(f"WARNING check needed — {col}: {test_df[col].isna().sum()} NaN in test after category alignment")

print("\nCategorical dtypes set:")
print(train_df[categorical_cols_final].dtypes)

WARNING check needed — consciousness_source: 1 NaN in test after category alignment

Categorical dtypes set:
gender                  category
race                    category
arrival_transport       category
avpu                    category
consciousness_source    category
complaint_category      category
dtype: object


### Phase 3 Summary

Starting from Phase 2's cleaned data (leakage-aware, NaN-preserving, categorically
labeled), this phase adds:

- 8 vital abnormality flags (clinical thresholds)
- 4 vital instability range features
- 1 consolidated consciousness flag
- 1 total-missingness count
- 1 bucketed complaint category (replacing the raw 175-value field)
- Finalized categorical dtypes ready for modeling

No feature here uses the target column or any test-set-derived statistic — every
threshold is a fixed clinical constant, and the only thing "fit" on train (the category
list in Step 6) is applied to test without peeking at test's own values.

**Next:** Phase 4 — train/validation split.